In [51]:
# V8-01 — load linking_v5 structural CSV and collapse duplicate roi_dir rows

from pathlib import Path
import pandas as pd
import numpy as np
import re

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 240)

ROOT = Path("/Users/davekokel/Projects/carp_v2")
if not ROOT.exists():
    raise FileNotFoundError(f"Expected repo root at {ROOT}, but it does not exist.")

BASE    = ROOT / "seed_kits" / "legacy_wrangling_v2"
RAW     = BASE / "raw"
WORKING = BASE / "working"

STRUCT_PATH = WORKING / "output_from_linking_v5.csv"

print("V8-01 — ROOT:      ", ROOT)
print("V8-01 — STRUCT_PATH:", STRUCT_PATH)

if not STRUCT_PATH.exists():
    raise FileNotFoundError(f"V8-01: Structural CSV not found at {STRUCT_PATH}")

df_struct_raw = pd.read_csv(STRUCT_PATH)

print("\nV8-01 — df_struct_raw shape:", df_struct_raw.shape)
print("V8-01 — df_struct_raw columns:")
print(list(df_struct_raw.columns))

print("\nV8-01 — basic ROI QC (raw):")
print("  total rows:      ", len(df_struct_raw))
print("  unique roi_dir:  ", df_struct_raw["roi_dir"].nunique())

dups = df_struct_raw[df_struct_raw.duplicated("roi_dir", keep=False)].copy()
print("  duplicated roi_dir rows:", len(dups))

# Treat some imaging fields as "noise-ish" that may differ across duplicated roi_dir,
# but should be UNIONed rather than treated as conflict.
UNION_COLS = [
    "additional plasmids injected",
    "additional mRNAs injected",
    "additonal proteins injected",
    "additonal dye and chemicals",
    "Imaged Locations",
    "Unique Targets with blanks",
    "Unique Targets",
]

KEEP_FIRST_COLS = [
    c for c in df_struct_raw.columns
    if c not in UNION_COLS
]

def _union_nonempty(series: pd.Series) -> str | None:
    vals = [
        str(x).strip()
        for x in series
        if pd.notna(x) and str(x).strip() not in ("", "nan", "None", "<NA>")
    ]
    if not vals:
        return None
    # explode comma/pipe lists if present, then dedupe
    tokens = []
    for v in vals:
        parts = re.split(r"[|,]", v)
        tokens.extend(p.strip() for p in parts if p.strip())
    uniq = []
    seen = set()
    for t in tokens:
        if t not in seen:
            seen.add(t)
            uniq.append(t)
    return "|".join(uniq) if uniq else None

if len(dups):
    # Group by roi_dir and aggregate
    grouped = df_struct_raw.groupby("roi_dir", dropna=False)

    records = []
    for roi_dir, grp in grouped:
        row = {}
        # columns we keep from the first occurrence
        first = grp.iloc[0]
        for c in KEEP_FIRST_COLS:
            row[c] = first.get(c, pd.NA)

        # union for noisy imaging columns
        for c in UNION_COLS:
            if c not in grp.columns:
                continue
            row[c] = _union_nonempty(grp[c])

        records.append(row)

    df_struct = pd.DataFrame.from_records(records)

    print("\nV8-01 — df_struct AFTER union-collapse:")
    print("  total rows:", len(df_struct))
    print("  unique roi_dir:", df_struct["roi_dir"].nunique())

    # sanity peek for mem-organelle rows
    mem_org = df_struct[df_struct["experiment_folder"] == "20250808_mem_organelle"].head(10)
    if not mem_org.empty:
        print("\nV8-01 — sample mem-organelle rows AFTER collapse:")
        print(
            mem_org[
                [
                    "roi_dir",
                    "experiment_folder",
                    "additional mRNAs injected",
                    "Unique Targets with blanks",
                ]
            ]
        )
else:
    df_struct = df_struct_raw.copy()
    print("\nV8-01 — no duplicate roi_dir rows; using raw df_struct_raw as df_struct.")

# Carry forward: df_struct is the structural base for V8

V8-01 — ROOT:       /Users/davekokel/Projects/carp_v2
V8-01 — STRUCT_PATH: /Users/davekokel/Projects/carp_v2/seed_kits/legacy_wrangling_v2/working/output_from_linking_v5.csv

V8-01 — df_struct_raw shape: (1082, 46)
V8-01 — df_struct_raw columns:
['date_experiment', 'fish', 'roi_rel', 'roi_name', 'roi_tiffs', 'roi_dir', 'dataset', 'experiment_folder', 'roi_path_date_yyyymmdd', 'fish_folder', 'roi_folder', 'fish_id', 'fish_number', 'fish_age_hpf', 'fish_nickname', 'fish_raw_norm', 'roi_anatomy_tokens', 'roi_anatomy', 'fish_folder_patched', 'dataset_slug', 'dataset_slug_norm', 'sheet_slug_norm', 'date_mount_yyyymmdd', 'date_mount', 'Date imaged', 'mount_id', 'ZF female genotype', 'ZF male genotype', 'additional plasmids injected', 'additional mRNAs injected', 'additonal proteins injected', 'additonal dye and chemicals', 'Date born', 'Imaged Locations', 'Unique Targets with blanks', 'Unique Targets', 'Data location', 'link_source', 'plate_date', 'mount_id_inferred', 'mount_id_source', 'pla

In [52]:
# V8-01b — seed treatment_rna_names_sheet / treatment_plasmid_names_sheet from imaging columns

df_enrich = df_struct.copy()

# Normalize text treatment columns to string dtype
for col in [
    "additional mRNAs injected",
    "additional plasmids injected",
]:
    if col in df_enrich.columns:
        df_enrich[col] = df_enrich[col].astype("string")

# Seed treatment_*_names_sheet if not already present
if "treatment_rna_names_sheet" not in df_enrich.columns:
    df_enrich["treatment_rna_names_sheet"] = pd.Series([pd.NA] * len(df_enrich), dtype="string")
else:
    df_enrich["treatment_rna_names_sheet"] = df_enrich["treatment_rna_names_sheet"].astype("string")

if "treatment_plasmid_names_sheet" not in df_enrich.columns:
    df_enrich["treatment_plasmid_names_sheet"] = pd.Series([pd.NA] * len(df_enrich), dtype="string")
else:
    df_enrich["treatment_plasmid_names_sheet"] = df_enrich["treatment_plasmid_names_sheet"].astype("string")

def _is_empty_str(series: pd.Series) -> pd.Series:
    return series.isna() | (series.astype(str).str.strip().isin(["", "None", "nan", "<NA>"]))

# Per-ROI seeding:
mask_rna_missing = _is_empty_str(df_enrich["treatment_rna_names_sheet"])
if "additional mRNAs injected" in df_enrich.columns:
    src = df_enrich["additional mRNAs injected"]
    mask_rna_has = src.notna() & (src.astype(str).str.strip() != "")
    to_seed_rna = mask_rna_missing & mask_rna_has
    df_enrich.loc[to_seed_rna, "treatment_rna_names_sheet"] = src[to_seed_rna]
else:
    to_seed_rna = pd.Series([False] * len(df_enrich))

mask_plasmid_missing = _is_empty_str(df_enrich["treatment_plasmid_names_sheet"])
if "additional plasmids injected" in df_enrich.columns:
    srcp = df_enrich["additional plasmids injected"]
    mask_plasmid_has = srcp.notna() & (srcp.astype(str).str.strip() != "")
    to_seed_plasmid = mask_plasmid_missing & mask_plasmid_has
    df_enrich.loc[to_seed_plasmid, "treatment_plasmid_names_sheet"] = srcp[to_seed_plasmid]
else:
    to_seed_plasmid = pd.Series([False] * len(df_enrich))

print("\nV8-01b — seeded treatment name columns from imaging:")
print(
    df_enrich[
        [
            "roi_dir",
            "experiment_folder",
            "additional mRNAs injected",
            "treatment_rna_names_sheet",
            "additional plasmids injected",
            "treatment_plasmid_names_sheet",
        ]
    ].head(20)
)

# Small mem-organelle sanity
mem_org = df_enrich[df_enrich["experiment_folder"] == "20250808_mem_organelle"].head(10)
if not mem_org.empty:
    print("\nV8-01b — mem-organelle rows after seeding:")
    print(
        mem_org[
            [
                "roi_dir",
                "experiment_folder",
                "additional mRNAs injected",
                "treatment_rna_names_sheet",
                "Unique Targets with blanks",
            ]
        ]
    )

print("\nV8-01b — df_enrich shape:", df_enrich.shape)


V8-01b — seeded treatment name columns from imaging:
                                              roi_dir                          experiment_folder additional mRNAs injected treatment_rna_names_sheet additional plasmids injected treatment_plasmid_names_sheet
0   /clusterfs/vast/abcabc/Aang_Foundation/2025072...  20250721_72hpf_mrna_mSG_organelle_LLS-SIM                      <NA>                      <NA>                         <NA>                          <NA>
1   /clusterfs/vast/abcabc/Aang_Foundation/2025072...  20250721_72hpf_mrna_mSG_organelle_LLS-SIM                      <NA>                      <NA>                         <NA>                          <NA>
2   /clusterfs/vast/abcabc/Aang_Foundation/2025072...  20250721_72hpf_mrna_mSG_organelle_LLS-SIM                      <NA>                      <NA>                         <NA>                          <NA>
3   /clusterfs/vast/abcabc/Aang_Foundation/2025072...  20250721_72hpf_mrna_mSG_organelle_LLS-SIM                  

In [53]:
# V8-02 — load parent map + injected RNA / plasmid catalogs and build mapping tables

from pathlib import Path
import pandas as pd
import re

if "df_enrich" not in globals():
    raise NameError("V8-02: df_enrich not found; run V8-01/V8-01b first.")

ROOT = Path("/Users/davekokel/Projects/carp_v2")
BASE = ROOT / "seed_kits" / "legacy_wrangling_v2"
RAW  = BASE / "raw"
AUTO = ROOT / "seed_kits" / "2025-11-15-121231-autoload"

PARENT_MAP_XLSX = RAW  / "Unique_parent_names__mom_dad_combined__preview_dqm_v5.xlsx"
PARENT_MAP_CSV  = RAW  / "Unique_parent_names__mom_dad_combined__preview_dqm_v5.csv"
INJECTED_RNA_PATH     = RAW / "Unique_injected_rna__preview_dqm.xlsx"
INJECTED_PLASMID_PATH = RAW / "Unique_injected_plasmid__preview_dqm.xlsx"

CONSTRUCTS_PATH = AUTO / "constructs_plasmid.csv"
TAGS_PATH       = AUTO / "tags.xlsx"
ALIAS_PATH      = AUTO / "alias.csv"

print("V8-02 — PARENT_MAP_XLSX:", PARENT_MAP_XLSX)
print("V8-02 — PARENT_MAP_CSV: ", PARENT_MAP_CSV)
print("V8-02 — INJECTED_RNA:   ", INJECTED_RNA_PATH)
print("V8-02 — INJECTED_PLASMID:", INJECTED_PLASMID_PATH)
print("V8-02 — CONSTRUCTS_PATH:", CONSTRUCTS_PATH)
print("V8-02 — TAGS_PATH:      ", TAGS_PATH)
print("V8-02 — ALIAS_PATH:     ", ALIAS_PATH)

# choose parent map loader
if PARENT_MAP_XLSX.exists():
    parent_map = pd.read_excel(PARENT_MAP_XLSX)
    print("V8-02 — parent_map loaded from XLSX")
elif PARENT_MAP_CSV.exists():
    parent_map = pd.read_csv(PARENT_MAP_CSV)
    print("V8-02 — parent_map loaded from CSV")
else:
    raise FileNotFoundError("V8-02: parent_map v5 not found as XLSX or CSV in RAW.")

missing = [p for p in [
    INJECTED_RNA_PATH,
    INJECTED_PLASMID_PATH,
    CONSTRUCTS_PATH,
    TAGS_PATH,
    ALIAS_PATH,
] if not p.exists()]

if missing:
    print("\nV8-02 — MISSING catalog path(s):")
    for p in missing:
        print("   ", p)
    raise FileNotFoundError("V8-02: some required catalogs are missing (see above).")

injected_rna     = pd.read_excel(INJECTED_RNA_PATH)
injected_plasmid = pd.read_excel(INJECTED_PLASMID_PATH)
constructs       = pd.read_csv(CONSTRUCTS_PATH)
tags_cat         = pd.read_excel(TAGS_PATH)
alias            = pd.read_csv(ALIAS_PATH)

print("\nV8-02 — shapes:")
print("  parent_map:      ", parent_map.shape)
print("  injected_rna:    ", injected_rna.shape)
print("  injected_plasmid:", injected_plasmid.shape)
print("  constructs:      ", constructs.shape)
print("  tags_cat:        ", tags_cat.shape)
print("  alias:           ", alias.shape)

print("\nV8-02 — parent_map columns:", list(parent_map.columns))
print("V8-02 — injected_rna columns:", list(injected_rna.columns))
print("V8-02 — injected_plasmid columns:", list(injected_plasmid.columns))

# normalize parent_map to expected columns:
# ['parent_fish_name', 'plasmid_base_code', 'allele', 'injected_rna', 'injected_plasmid']
parent_cols_norm = {
    "parent_fish_name": "parent_fish_name",
    "parent_name": "parent_fish_name",
    "plasmid_base_code": "plasmid_base_code",
    "allele": "allele",
    "injected_rna": "injected_rna",
    "injected_rna(s)": "injected_rna",
    "injected plasmid": "injected_plasmid",
    "injected_plasmid": "injected_plasmid",
}
parent_map = parent_map.rename(columns={k: v for k, v in parent_cols_norm.items() if k in parent_map.columns})

needed_parent_cols = ["parent_fish_name", "plasmid_base_code", "allele", "injected_rna", "injected_plasmid"]
missing_parent_cols = [c for c in needed_parent_cols if c not in parent_map.columns]
if missing_parent_cols:
    raise KeyError(f"V8-02: parent_map missing expected columns: {missing_parent_cols}")

print("\nV8-02 — parent_map normalized columns:", list(parent_map.columns))

# helper to normalize labels
def _norm_label(s: str | float | None) -> str | None:
    if pd.isna(s):
        return None
    s = str(s).strip().lower()
    # strip leading YYYYMMDD_ or YYYYMMDD-
    s = re.sub(r"^\d{8}[_-]", "", s)
    # drop parenthetical "(F1 of allele 325)" etc
    s = re.sub(r"\(.*?\)", "", s)
    # collapse non-alphanumeric
    s = re.sub(r"[^a-z0-9]+", "", s)
    s = s.strip()
    return s or None

# build parent_slug_norm in parent_map
parent_map["parent_slug_norm"] = parent_map["parent_fish_name"].apply(_norm_label)

# Build parent_slug_agg: slug → genotype + injected RNA names
def _agg_nonempty(series: pd.Series) -> str | None:
    vals = [
        str(x).strip()
        for x in series
        if pd.notna(x) and str(x).strip() not in ("", "nan", "None", "<NA>")
    ]
    if not vals:
        return None
    tokens = []
    for v in vals:
        parts = re.split(r"[|,]", v)
        tokens.extend(p.strip() for p in parts if p.strip())
    seen = set()
    out = []
    for t in tokens:
        if t not in seen:
            seen.add(t)
            out.append(t)
    return "|".join(out) if out else None

parent_slug_agg = (
    parent_map
    .groupby("parent_slug_norm", dropna=True)
    .agg({
        "plasmid_base_code": _agg_nonempty,
        "allele": _agg_nonempty,
        "injected_rna": _agg_nonempty,
        "injected_plasmid": _agg_nonempty,
    })
    .reset_index()
    .rename(columns={
        "plasmid_base_code": "geno_base_codes_v8_map",
        "allele": "geno_alleles_v8_map",
        "injected_rna": "inj_rna_names_v8_map",
        "injected_plasmid": "inj_plasmid_names_v8_map",
    })
)

print("\nV8-02 — parent_slug_agg sample:")
print(parent_slug_agg.head(20))

# Build injected RNA map: treatment_name_norm → rna_base_code
rna_df = injected_rna.copy()
rna_name_col = None
rna_base_col = None
for c in rna_df.columns:
    cl = c.strip().lower()
    if "injected_rna" in cl or cl == "injected_rna":
        rna_name_col = c
    if "plasmid_base_code" in cl:
        rna_base_col = c

if not rna_name_col or not rna_base_col:
    raise KeyError("V8-02: cannot detect injected_rna name/basecode columns in injected_rna sheet.")

rna_map = (
    rna_df[[rna_name_col, rna_base_col]]
    .rename(columns={rna_name_col: "treatment_name", rna_base_col: "rna_base_code"})
    .dropna(subset=["treatment_name", "rna_base_code"])
    .copy()
)

def _norm_treatment_name(s):
    if pd.isna(s):
        return None
    s = str(s).strip()
    return s or None

rna_map["treatment_name_norm"] = rna_map["treatment_name"].apply(_norm_treatment_name)
rna_map = rna_map.drop_duplicates("treatment_name_norm")

print("\nV8-02 — rna_map sample:")
print(rna_map.head(10))

# Build injected plasmid map: treatment_name_norm → plasmid_base_code
pl_df = injected_plasmid.copy()
pl_name_col = None
pl_base_col = None
for c in pl_df.columns:
    cl = c.strip().lower()
    if "injected_plasmid" in cl or "plasmid" == cl:
        if pl_name_col is None:
            pl_name_col = c
    if "plasmid_base_code" in cl:
        pl_base_col = c

if not pl_name_col or not pl_base_col:
    plasmid_map = pd.DataFrame(columns=["treatment_name_norm", "plasmid_base_code"])
else:
    plasmid_map = (
        pl_df[[pl_name_col, pl_base_col]]
        .rename(columns={pl_name_col: "treatment_name", pl_base_col: "plasmid_base_code"})
        .dropna(subset=["treatment_name", "plasmid_base_code"])
        .copy()
    )
    plasmid_map["treatment_name_norm"] = plasmid_map["treatment_name"].apply(_norm_treatment_name)
    plasmid_map = plasmid_map.drop_duplicates("treatment_name_norm")

print("\nV8-02 — plasmid_map sample:")
print(plasmid_map.head(10))

print("\nV8-02 — catalogs and maps ready. Next: V8-03b (parent-based patches) and V8-03c (experiment_hole_patch_v7).")

V8-02 — PARENT_MAP_XLSX: /Users/davekokel/Projects/carp_v2/seed_kits/legacy_wrangling_v2/raw/Unique_parent_names__mom_dad_combined__preview_dqm_v5.xlsx
V8-02 — PARENT_MAP_CSV:  /Users/davekokel/Projects/carp_v2/seed_kits/legacy_wrangling_v2/raw/Unique_parent_names__mom_dad_combined__preview_dqm_v5.csv
V8-02 — INJECTED_RNA:    /Users/davekokel/Projects/carp_v2/seed_kits/legacy_wrangling_v2/raw/Unique_injected_rna__preview_dqm.xlsx
V8-02 — INJECTED_PLASMID: /Users/davekokel/Projects/carp_v2/seed_kits/legacy_wrangling_v2/raw/Unique_injected_plasmid__preview_dqm.xlsx
V8-02 — CONSTRUCTS_PATH: /Users/davekokel/Projects/carp_v2/seed_kits/2025-11-15-121231-autoload/constructs_plasmid.csv
V8-02 — TAGS_PATH:       /Users/davekokel/Projects/carp_v2/seed_kits/2025-11-15-121231-autoload/tags.xlsx
V8-02 — ALIAS_PATH:      /Users/davekokel/Projects/carp_v2/seed_kits/2025-11-15-121231-autoload/alias.csv
V8-02 — parent_map loaded from CSV

V8-02 — shapes:
  parent_map:       (52, 5)
  injected_rna:    

In [54]:
# V8-03b — parent-based patch: genotype + treatment_rna_names_sheet from parent_map

import pandas as pd
import re

if "df_enrich" not in globals():
    raise NameError("V8-03b: df_enrich not found; run V8-01/V8-01b/V8-02 first.")
if "parent_slug_agg" not in globals():
    raise NameError("V8-03b: parent_slug_agg not found; run V8-02 first.")

df_enrich = df_enrich.copy()

# ─────────────────────────────────────────────
# 1) exp_slug_norm on df_enrich (from experiment_folder)
# ─────────────────────────────────────────────

def _norm_label_v8(s):
    if pd.isna(s):
        return None
    s = str(s).strip().lower()
    s = re.sub(r"^\d{8}[_-]", "", s)     # strip leading YYYYMMDD_
    s = re.sub(r"\(.*?\)", "", s)        # drop "(F1 of allele 325)"
    s = re.sub(r"[^a-z0-9]+", "", s)     # collapse whitespace/punct
    s = s.strip()
    return s or None

df_enrich["exp_slug_norm"] = df_enrich["experiment_folder"].apply(_norm_label_v8)

print("V8-03b — exp_slug_norm unique:", df_enrich["exp_slug_norm"].nunique())
print("V8-03b — parent_slug_norm unique:", parent_slug_agg["parent_slug_norm"].nunique())

# ─────────────────────────────────────────────
# 2) Join parent_slug_agg onto df_enrich
# ─────────────────────────────────────────────

merge_cols = [
    "parent_slug_norm",
    "geno_base_codes_v8_map",
    "geno_alleles_v8_map",
    "inj_rna_names_v8_map",
    "inj_plasmid_names_v8_map",
]

df_enrich = df_enrich.merge(
    parent_slug_agg[merge_cols],
    how="left",
    left_on="exp_slug_norm",
    right_on="parent_slug_norm",
)

# ensure target cols are string
for col in [
    "genotype_base_codes",
    "genotype_allele_codes",
    "treatment_rna_names_sheet",
    "treatment_plasmid_names_sheet",
]:
    if col not in df_enrich.columns:
        df_enrich[col] = pd.Series([pd.NA] * len(df_enrich), dtype="string")
    else:
        df_enrich[col] = df_enrich[col].astype("string")

def _is_empty_str(s: pd.Series) -> pd.Series:
    return s.isna() | (s.astype(str).str.strip().isin(["", "None", "nan", "<NA>"]))

# ─────────────────────────────────────────────
# 3) Patch genotype from parent map
# ─────────────────────────────────────────────

mask_geno_missing = _is_empty_str(df_enrich["genotype_base_codes"])
mask_allele_missing = _is_empty_str(df_enrich["genotype_allele_codes"])

mask_geno_has_parent   = df_enrich["geno_base_codes_v8_map"].notna()
mask_allele_has_parent = df_enrich["geno_alleles_v8_map"].notna()

to_patch_geno   = mask_geno_missing   & mask_geno_has_parent
to_patch_allele = mask_allele_missing & mask_allele_has_parent

df_enrich["geno_base_codes_v8_map"] = df_enrich["geno_base_codes_v8_map"].astype("string")
df_enrich["geno_alleles_v8_map"]    = df_enrich["geno_alleles_v8_map"].astype("string")

df_enrich.loc[to_patch_geno,   "genotype_base_codes"]   = df_enrich.loc[to_patch_geno,   "geno_base_codes_v8_map"]
df_enrich.loc[to_patch_allele, "genotype_allele_codes"] = df_enrich.loc[to_patch_allele, "geno_alleles_v8_map"]

print("\nV8-03b — genotype patch summary:")
print("  geno_missing before patch:", int(mask_geno_missing.sum()))
print("  geno_parent slugs present:", int(mask_geno_has_parent.sum()))
print("  patched genotype rows:    ", int(to_patch_geno.sum()))

# ─────────────────────────────────────────────
# 4) Patch treatment RNA names from parent inj_rna_names_v8_map
# ─────────────────────────────────────────────

df_enrich["inj_rna_names_v8_map"] = df_enrich["inj_rna_names_v8_map"].astype("string")
df_enrich["inj_plasmid_names_v8_map"] = df_enrich["inj_plasmid_names_v8_map"].astype("string")

mask_rna_missing   = _is_empty_str(df_enrich["treatment_rna_names_sheet"])
mask_pl_missing    = _is_empty_str(df_enrich["treatment_plasmid_names_sheet"])
mask_rna_has_parent = df_enrich["inj_rna_names_v8_map"].notna()
mask_pl_has_parent  = df_enrich["inj_plasmid_names_v8_map"].notna()

to_patch_rna = mask_rna_missing & mask_rna_has_parent
to_patch_pl  = mask_pl_missing  & mask_pl_has_parent

df_enrich.loc[to_patch_rna, "treatment_rna_names_sheet"] = df_enrich.loc[to_patch_rna, "inj_rna_names_v8_map"]
df_enrich.loc[to_patch_pl,  "treatment_plasmid_names_sheet"] = df_enrich.loc[to_patch_pl, "inj_plasmid_names_v8_map"]

print("\nV8-03b — treatment name patch summary:")
print("  rna_missing before:", int(mask_rna_missing.sum()))
print("  rna_parent slugs:  ", int(mask_rna_has_parent.sum()))
print("  patched RNA names: ", int(to_patch_rna.sum()))
print("  patched plasmid names:", int(to_patch_pl.sum()))

# quick sanity peek for some key parent slugs
for needle in ["skittle", "memmito", "memhistone", "peroxi"]:
    sub = df_enrich[df_enrich["exp_slug_norm"].fillna("").str.contains(needle, na=False)]
    if not sub.empty:
        print(f"\nV8-03b — sample rows for exp_slug_norm containing '{needle}':")
        print(
            sub[
                [
                    "roi_dir",
                    "experiment_folder",
                    "exp_slug_norm",
                    "genotype_base_codes",
                    "genotype_allele_codes",
                    "treatment_rna_names_sheet",
                ]
            ].head(10)
        )

print("\nV8-03b — parent-based patch done. Next: V8-03c (experiment_hole_patch_v7).")

V8-03b — exp_slug_norm unique: 32
V8-03b — parent_slug_norm unique: 41

V8-03b — genotype patch summary:
  geno_missing before patch: 976
  geno_parent slugs present: 767
  patched genotype rows:     767

V8-03b — treatment name patch summary:
  rna_missing before: 292
  rna_parent slugs:   372
  patched RNA names:  93
  patched plasmid names: 0

V8-03b — sample rows for exp_slug_norm containing 'skittle':
                                               roi_dir  experiment_folder exp_slug_norm genotype_base_codes genotype_allele_codes treatment_rna_names_sheet
566  /clusterfs/vast/abcabc/Korra_Foundation/202505...  20250513_skittles      skittles             pDQM034                   309                  phiC-NLS
567  /clusterfs/vast/abcabc/Korra_Foundation/202505...  20250513_skittles      skittles             pDQM034                   309                  phiC-NLS
568  /clusterfs/vast/abcabc/Korra_Foundation/202505...  20250513_skittles      skittles             pDQM034               

In [55]:
# V8-03c — experiment-level slug→basecode patch via CSV (experiment_hole_patch_v7)

from pathlib import Path
import pandas as pd

if "df_enrich" not in globals():
    raise NameError("V8-03c: df_enrich not found; run V8-01..V8-03b first.")

ROOT = Path("/Users/davekokel/Projects/carp_v2")
BASE = ROOT / "seed_kits" / "legacy_wrangling_v2"
RAW  = BASE / "raw"

EXP_PATCH_PATH = RAW / "experiment_hole_patch_v7.csv"
print("V8-03c — EXP_PATCH_PATH:", EXP_PATCH_PATH)

if not EXP_PATCH_PATH.exists():
    raise FileNotFoundError(f"V8-03c: experiment hole patch CSV not found at {EXP_PATCH_PATH}")

exp_patch = pd.read_csv(EXP_PATCH_PATH)

required_cols = {
    "dataset_slug",
    "genotype_base_codes",
    "genotype_allele_codes",
    "treatment_rna_base_codes",
    "treatment_plasmid_base_codes",
}
missing = required_cols - set(exp_patch.columns)
if missing:
    raise KeyError(f"V8-03c: experiment_hole_patch_v7.csv missing columns: {sorted(missing)}")

def _norm_slug(s):
    if pd.isna(s):
        return None
    return str(s).strip()

exp_patch = exp_patch.copy()
exp_patch["dataset_slug_norm"] = exp_patch["dataset_slug"].apply(_norm_slug)

df_enrich = df_enrich.copy()
df_enrich["dataset_slug_norm"] = df_enrich.get("dataset_slug_norm", df_enrich["dataset_slug"].apply(_norm_slug))

print("V8-03c — exp_patch rows:", len(exp_patch))
print("V8-03c — exp_patch unique dataset_slug_norm:", exp_patch["dataset_slug_norm"].nunique())

# mapping from patch CSV → df_enrich columns
patch_map = {
    "genotype_base_codes":          "genotype_base_codes",
    "genotype_allele_codes":        "genotype_allele_codes",
    "treatment_rna_base_codes":     "treatment_rna_rna_base_code",
    "treatment_plasmid_base_codes": "treatment_plasmid_plasmid_base_code",
}

# ensure target cols exist and string-typed
for tgt_col in patch_map.values():
    if tgt_col not in df_enrich.columns:
        df_enrich[tgt_col] = pd.Series([pd.NA] * len(df_enrich), dtype="string")
    else:
        df_enrich[tgt_col] = df_enrich[tgt_col].astype("string")

merge_cols = ["dataset_slug_norm"] + list(patch_map.keys())
patch_small = (
    exp_patch[merge_cols]
    .drop_duplicates("dataset_slug_norm")
)

df_enrich = df_enrich.merge(
    patch_small,
    how="left",
    on="dataset_slug_norm",
    suffixes=("", "_exp_patch"),
)

def _is_empty_str(series: pd.Series) -> pd.Series:
    return series.isna() | (series.astype(str).str.strip().isin(["", "None", "nan", "<NA>"]))

print("V8-03c — patchable columns:", list(patch_map.values()))

for src_col, tgt_col in patch_map.items():
    patch_col = f"{src_col}_exp_patch"
    if patch_col not in df_enrich.columns:
        continue

    current = df_enrich[tgt_col]
    patch_vals = df_enrich[patch_col].astype("string")

    mask_missing = _is_empty_str(current)
    mask_has_patch = patch_vals.notna() & (patch_vals.astype(str).str.strip() != "")

    to_patch = mask_missing & mask_has_patch

    print(f"\nV8-03c — patch for {tgt_col}:")
    print("  missing before:", int(mask_missing.sum()))
    print("  rows with patch:", int(mask_has_patch.sum()))
    print("  patched:", int(to_patch.sum()))

    df_enrich.loc[to_patch, tgt_col] = patch_vals[to_patch]

# clean up helper columns
drop_cols = [c for c in df_enrich.columns if c.endswith("_exp_patch")]
df_enrich = df_enrich.drop(columns=drop_cols)

must_be_full_slugs = [
    "20250805_lifeact_mem-halo",
    "20251028_mem-peroxi",
    "20251028_mem-peroxi2",
    "20250521_skittles_no-membrane",
    "20251017_nuclear_envelope",
    "20251107_mem-kinectocore",
    "20250602_mem",
    "20251017_microtubules",
]

print("\nV8-03c — sample rows after experiment-level patch (must-be-full slugs):")
print(
    df_enrich[
        df_enrich["dataset_slug"].isin(must_be_full_slugs)
    ][[
        "dataset_slug",
        "roi_dir",
        "genotype_base_codes",
        "genotype_allele_codes",
        "treatment_rna_rna_base_code",
        "treatment_plasmid_plasmid_base_code",
    ]].head(40)
)

print("\nV8-03c — done. Next: V8-02b (token-wise text→basecode for treatments) and then V8-04 (constructs_ft + marker rollup).")

V8-03c — EXP_PATCH_PATH: /Users/davekokel/Projects/carp_v2/seed_kits/legacy_wrangling_v2/raw/experiment_hole_patch_v7.csv
V8-03c — exp_patch rows: 13
V8-03c — exp_patch unique dataset_slug_norm: 12
V8-03c — patchable columns: ['genotype_base_codes', 'genotype_allele_codes', 'treatment_rna_rna_base_code', 'treatment_plasmid_plasmid_base_code']

V8-03c — patch for genotype_base_codes:
  missing before: 209
  rows with patch: 18
  patched: 18

V8-03c — patch for genotype_allele_codes:
  missing before: 209
  rows with patch: 18
  patched: 18

V8-03c — sample rows after experiment-level patch (must-be-full slugs):
                      dataset_slug                                            roi_dir genotype_base_codes genotype_allele_codes treatment_rna_rna_base_code treatment_plasmid_plasmid_base_code
27       20250805_lifeact_mem-halo  /clusterfs/vast/abcabc/Aang_Foundation/2025080...                <NA>                  <NA>                        <NA>                                <NA

In [56]:
# V8-03d — token-wise treatment text → basecode mapping (RNA + plasmid)

import re
import pandas as pd

if "df_enrich" not in globals():
    raise NameError("V8-03d: df_enrich not found; run V8-01..V8-03c first.")
if "rna_map" not in globals() or "plasmid_map" not in globals():
    raise NameError("V8-03d: rna_map/plasmid_map not found; run V8-02 first.")

df_enrich = df_enrich.copy()

# ─────────────────────────────────────────────
# 1) Normalize existing treatment name columns
# ─────────────────────────────────────────────

def _norm_treatment_name_v8(s: str | float | None) -> str | None:
    if pd.isna(s):
        return None
    s = str(s).strip()
    return s or None

for col in ["treatment_rna_names_sheet", "treatment_plasmid_names_sheet"]:
    if col not in df_enrich.columns:
        df_enrich[col] = pd.Series([pd.NA] * len(df_enrich), dtype="string")
    else:
        df_enrich[col] = df_enrich[col].astype("string")

df_enrich["treatment_rna_names_norm"] = df_enrich["treatment_rna_names_sheet"].apply(_norm_treatment_name_v8)
df_enrich["treatment_plasmid_names_norm"] = df_enrich["treatment_plasmid_names_sheet"].apply(_norm_treatment_name_v8)

# current basecode cols as string
for col in ["treatment_rna_rna_base_code", "treatment_plasmid_plasmid_base_code"]:
    if col not in df_enrich.columns:
        df_enrich[col] = pd.Series([pd.NA] * len(df_enrich), dtype="string")
    else:
        df_enrich[col] = df_enrich[col].astype("string")

# ─────────────────────────────────────────────
# 2) Helpers: tokenization + lookup
# ─────────────────────────────────────────────

# create dicts for quick lookup
rna_dict = {
    str(row["treatment_name_norm"]): str(row["rna_base_code"])
    for _, row in rna_map.dropna(subset=["treatment_name_norm", "rna_base_code"]).iterrows()
}

pl_dict = {}
if "plasmid_base_code" in plasmid_map.columns:
    pl_dict = {
        str(row["treatment_name_norm"]): str(row["plasmid_base_code"])
        for _, row in plasmid_map.dropna(subset=["treatment_name_norm", "plasmid_base_code"]).iterrows()
    }

# basecodes we recognize directly (MGCO-xx, pDQMxxx, etc.)
basecode_set = set(rna_map["rna_base_code"].dropna().astype(str)) | set(
    plasmid_map.get("plasmid_base_code", pd.Series([], dtype="string")).dropna().astype(str)
)

def _split_tokens(val: str | float | None) -> list[str]:
    if pd.isna(val):
        return []
    s = str(val)
    # split on common separators: | , ; and " + "
    parts = re.split(r"[|,;]", s)
    out = []
    for p in parts:
        p = p.strip()
        if p:
            out.append(p)
    return out

def _tokens_to_rna_basecodes(name_str: str | float | None) -> str | None:
    tokens = _split_tokens(name_str)
    codes: list[str] = []

    for tok in tokens:
        norm = _norm_treatment_name_v8(tok)
        if not norm:
            continue

        # 1) direct text → basecode via rna_map
        if norm in rna_dict:
            codes.append(rna_dict[norm])
            continue

        # 2) if token itself looks like a basecode we know, keep it
        if tok in basecode_set:
            codes.append(tok)
            continue

    if not codes:
        return None

    # dedupe but keep order
    seen = set()
    out = []
    for c in codes:
        if c not in seen:
            seen.add(c)
            out.append(c)
    return "|".join(out)

def _tokens_to_plasmid_basecodes(name_str: str | float | None) -> str | None:
    tokens = _split_tokens(name_str)
    codes: list[str] = []

    for tok in tokens:
        norm = _norm_treatment_name_v8(tok)
        if not norm:
            continue

        if norm in pl_dict:
            codes.append(pl_dict[norm])
            continue

        if tok in basecode_set:
            codes.append(tok)
            continue

    if not codes:
        return None

    seen = set()
    out = []
    for c in codes:
        if c not in seen:
            seen.add(c)
            out.append(c)
    return "|".join(out)

# ─────────────────────────────────────────────
# 3) Apply mapping only where basecodes are still missing
# ─────────────────────────────────────────────

def _is_empty_str(series: pd.Series) -> pd.Series:
    return series.isna() | (series.astype(str).str.strip().isin(["", "None", "nan", "<NA>"]))

# RNA
mask_rna_missing = _is_empty_str(df_enrich["treatment_rna_rna_base_code"])
rna_candidates = df_enrich.loc[mask_rna_missing, "treatment_rna_names_norm"]

rna_mapped = rna_candidates.apply(_tokens_to_rna_basecodes)

mask_rna_has_new = rna_mapped.notna()
df_enrich.loc[mask_rna_missing & mask_rna_has_new, "treatment_rna_rna_base_code"] = (
    rna_mapped[mask_rna_has_new].astype("string")
)

print("V8-03d — RNA mapping:")
print("  rows with missing basecodes before:", int(mask_rna_missing.sum()))
print("  rows where names→basecodes produced something:", int(mask_rna_has_new.sum()))

# Plasmid (we rarely use this for organelles right now, but fill it for completeness)
mask_pl_missing = _is_empty_str(df_enrich["treatment_plasmid_plasmid_base_code"])
pl_candidates = df_enrich.loc[mask_pl_missing, "treatment_plasmid_names_norm"]

pl_mapped = pl_candidates.apply(_tokens_to_plasmid_basecodes)
mask_pl_has_new = pl_mapped.notna()
df_enrich.loc[mask_pl_missing & mask_pl_has_new, "treatment_plasmid_plasmid_base_code"] = (
    pl_mapped[mask_pl_has_new].astype("string")
)

print("\nV8-03d — plasmid mapping:")
print("  rows with missing plasmid basecodes before:", int(mask_pl_missing.sum()))
print("  rows where names→basecodes produced something:", int(mask_pl_has_new.sum()))

# sanity: show a few of the tricky datasets
for slug in [
    "20250808_mem_organelle",
    "20250715_mem_organelle",
    "20250917_mem-mito",
    "20250513_skittles",
    "20250521_skittles_no-membrane",
]:
    sub = df_enrich[df_enrich["dataset_slug"] == slug]
    if sub.empty:
        continue
    print(f"\nV8-02b — sample rows after text→basecode for dataset_slug='{slug}':")
    print(
        sub[
            [
                "roi_dir",
                "dataset_slug",
                "treatment_rna_names_sheet",
                "treatment_rna_rna_base_code",
                "treatment_plasmid_names_sheet",
                "treatment_plasmid_plasmid_base_code",
            ]
        ].head(10)
    )

print("\nV8-03d — token-wise treatment mapping complete. Next: V8-04 (constructs_ft + marker rollup).")

V8-03d — RNA mapping:
  rows with missing basecodes before: 976
  rows where names→basecodes produced something: 547

V8-03d — plasmid mapping:
  rows with missing plasmid basecodes before: 976
  rows where names→basecodes produced something: 29

V8-02b — sample rows after text→basecode for dataset_slug='20250808_mem_organelle':
                                              roi_dir            dataset_slug treatment_rna_names_sheet treatment_rna_rna_base_code treatment_plasmid_names_sheet treatment_plasmid_plasmid_base_code
39  /clusterfs/vast/abcabc/Aang_Foundation/2025080...  20250808_mem_organelle     2xCox8A:mSG|LAMP1:mSG             MGCO-01|MGCO-15                          <NA>                                <NA>
40  /clusterfs/vast/abcabc/Aang_Foundation/2025080...  20250808_mem_organelle     2xCox8A:mSG|LAMP1:mSG             MGCO-01|MGCO-15                          <NA>                                <NA>
41  /clusterfs/vast/abcabc/Aang_Foundation/2025080...  20250808_mem_organel

In [57]:
# V8-04 — build constructs_ft (basecode → fluor/tag/localization) from constructs_plasmid.csv

import pandas as pd

if "constructs" not in globals():
    raise NameError("V8-04: 'constructs' not found; run V8-02 first.")

# We expect constructs to already have one row per (plasmid_code, fluor_code[, tag_code, tag_pos, tag_localization])
print("V8-04 — constructs columns:", list(constructs.columns))

required_cols_cf = {"plasmid_code", "fluor_code"}
missing_cf = required_cols_cf - set(constructs.columns)
if missing_cf:
    raise KeyError(f"V8-04: constructs is missing columns: {sorted(missing_cf)}")

# Optional tag columns
has_tag_code         = "tag_code" in constructs.columns
has_tag_localization = "tag_localization" in constructs.columns

cols_keep = ["plasmid_code", "fluor_code"]
if has_tag_code:
    cols_keep.append("tag_code")
if has_tag_localization:
    cols_keep.append("tag_localization")

constructs_ft = constructs[cols_keep].copy()

# normalize types
constructs_ft["plasmid_code"] = constructs_ft["plasmid_code"].astype("string").str.strip()
constructs_ft["fluor_code"]   = constructs_ft["fluor_code"].astype("string").str.strip()

if has_tag_code:
    constructs_ft["tag_code"] = constructs_ft["tag_code"].astype("string").str.strip()
else:
    constructs_ft["tag_code"] = pd.NA

if has_tag_localization:
    constructs_ft["tag_localization"] = constructs_ft["tag_localization"].astype("string").str.strip()
else:
    # Un-tagged constructs → organelle = cytosol by default
    constructs_ft["tag_localization"] = "cytosol"

print("\nV8-04 — constructs_ft sample:")
print(constructs_ft.head(20))
print("\nV8-04 — constructs_ft distinct plasmids:", constructs_ft["plasmid_code"].nunique())
print("V8-04 — constructs_ft distinct fluors:", constructs_ft["fluor_code"].nunique())
print("V8-04 — constructs_ft distinct tag_localizations:", constructs_ft["tag_localization"].dropna().unique())

V8-04 — constructs columns: ['plasmid_code', 'plasmid_name', 'plasmid_nickname', 'resistance', 'plasmid_notes', 'used_for_injection_plasmid', 'used_for_injection_rna', 'used_for_injection_crispr', 'n_fluors_per_plasmid', 'fluor_code', 'tag_code', 'tag_pos']

V8-04 — constructs_ft sample:
   plasmid_code fluor_code tag_code tag_localization
0       pDQM001        mSG     <NA>          cytosol
1       pDQM002        mSG     <NA>          cytosol
2       pDQM005      tdmSG   2xLynk          cytosol
3       pDQM006      tdmSG     <NA>          cytosol
4       pDQM007      tdmSG     <NA>          cytosol
5       pDQM008      tdmSG     <NA>          cytosol
6       pDQM009   mScarlet     <NA>          cytosol
7       pDQM009     mKate2     <NA>          cytosol
8       pDQM009   Electra2     <NA>          cytosol
9       pDQM009       mKOK     <NA>          cytosol
10      pDQM009      mTFP1     <NA>          cytosol
11      pDQM010   mScarlet     <NA>          cytosol
12      pDQM010     mK

In [58]:
# V8-05 — marker rollup and organelles from genotype + treatment basecodes

import re
import pandas as pd

if "df_enrich" not in globals():
    raise NameError("V8-05: df_enrich not found; run V8-01..V8-03d first.")
if "constructs_ft" not in globals():
    raise NameError("V8-05: constructs_ft not found; run V8-04 first.")

df_enrich = df_enrich.copy()

# Ensure basecode columns exist and are strings
for col in [
    "genotype_base_codes",
    "treatment_rna_rna_base_code",
    "treatment_plasmid_plasmid_base_code",
]:
    if col not in df_enrich.columns:
        df_enrich[col] = pd.Series([pd.NA] * len(df_enrich), dtype="string")
    else:
        df_enrich[col] = df_enrich[col].astype("string")

def _split_codes(val):
    if pd.isna(val):
        return []
    s = str(val).strip()
    if not s:
        return []
    parts = [p.strip() for p in re.split(r"[|,]", s) if p.strip()]
    return parts

# ───────── genotype markers ─────────

geno_rows = []
for _, row in df_enrich[["roi_dir", "genotype_base_codes"]].iterrows():
    codes = _split_codes(row["genotype_base_codes"])
    for code in codes:
        geno_rows.append((row["roi_dir"], code))

geno_df = pd.DataFrame(geno_rows, columns=["roi_dir", "plasmid_code"])

if geno_df.empty:
    geno_markers = pd.DataFrame(columns=[
        "roi_dir",
        "genotype_marker_fluor_codes",
        "genotype_marker_tag_codes",
        "genotype_marker_localizations",
        "genotype_marker_fusion_labels",
    ])
else:
    # normalize key for merge
    constructs_ft_local = constructs_ft.copy()
    constructs_ft_local["plasmid_code"] = constructs_ft_local["plasmid_code"].astype("string").str.strip()
    geno_df["plasmid_code"] = geno_df["plasmid_code"].astype("string").str.strip()

    geno_df = geno_df.merge(constructs_ft_local, on="plasmid_code", how="left")

    def _agg_geno(group):
        fl  = [f for f in group["fluor_code"]        if pd.notna(f)]
        tg  = [t for t in group["tag_code"]          if pd.notna(t)]
        loc = [l for l in group["tag_localization"]  if pd.notna(l)]

        def uniq(xs):
            out, seen = [], set()
            for x in xs:
                if x not in seen:
                    seen.add(x)
                    out.append(x)
            return out

        fl_u  = uniq(fl)
        tg_u  = uniq(tg)
        loc_u = uniq(loc)

        fusion_labels = []
        for f, l in zip(fl, loc):
            if pd.notna(f) and pd.notna(l):
                fusion_labels.append(f"{f}({l})")
        fusion_u = uniq(fusion_labels)

        return pd.Series({
            "genotype_marker_fluor_codes": "|".join(fl_u)  if fl_u  else pd.NA,
            "genotype_marker_tag_codes":   "|".join(tg_u)  if tg_u  else pd.NA,
            "genotype_marker_localizations": "|".join(loc_u) if loc_u else pd.NA,
            "genotype_marker_fusion_labels": "|".join(fusion_u) if fusion_u else pd.NA,
        })

    geno_markers = (
        geno_df.groupby("roi_dir", dropna=False)
        .apply(_agg_geno)
        .reset_index()
    )

# ───────── treatment markers (RNA + plasmid) ─────────

tx_rows = []
for _, row in df_enrich[["roi_dir", "treatment_rna_rna_base_code", "treatment_plasmid_plasmid_base_code"]].iterrows():
    for code in _split_codes(row["treatment_rna_rna_base_code"]):
        tx_rows.append((row["roi_dir"], code))
    for code in _split_codes(row["treatment_plasmid_plasmid_base_code"]):
        tx_rows.append((row["roi_dir"], code))

tx_df = pd.DataFrame(tx_rows, columns=["roi_dir", "plasmid_code"])

if tx_df.empty:
    tx_markers = pd.DataFrame(columns=[
        "roi_dir",
        "treatment_marker_fluor_codes",
        "treatment_marker_tag_codes",
        "treatment_marker_localizations",
        "treatment_marker_fluor_loc_labels",
    ])
else:
    constructs_ft_local = constructs_ft.copy()
    constructs_ft_local["plasmid_code"] = constructs_ft_local["plasmid_code"].astype("string").str.strip()
    tx_df["plasmid_code"] = tx_df["plasmid_code"].astype("string").str.strip()

    tx_df = tx_df.merge(constructs_ft_local, on="plasmid_code", how="left")

    def _agg_tx(group):
        fl  = [f for f in group["fluor_code"]        if pd.notna(f)]
        tg  = [t for t in group["tag_code"]          if pd.notna(t)]
        loc = [l for l in group["tag_localization"]  if pd.notna(l)]

        def uniq(xs):
            out, seen = [], set()
            for x in xs:
                if x not in seen:
                    seen.add(x)
                    out.append(x)
            return out

        fl_u  = uniq(fl)
        tg_u  = uniq(tg)
        loc_u = uniq(loc)

        fusion_labels = []
        for f, l in zip(fl, loc):
            if pd.notna(f) and pd.notna(l):
                fusion_labels.append(f"{f}({l})")
        fusion_u = uniq(fusion_labels)

        return pd.Series({
            "treatment_marker_fluor_codes": "|".join(fl_u)  if fl_u  else pd.NA,
            "treatment_marker_tag_codes":   "|".join(tg_u)  if tg_u  else pd.NA,
            "treatment_marker_localizations": "|".join(loc_u) if loc_u else pd.NA,
            "treatment_marker_fluor_loc_labels": "|".join(fusion_u) if fusion_u else pd.NA,
        })

    tx_markers = (
        tx_df.groupby("roi_dir", dropna=False)
        .apply(_agg_tx)
        .reset_index()
    )

# ───────── merge markers → df_enrich ─────────

for mdf in [geno_markers, tx_markers]:
    df_enrich = df_enrich.merge(mdf, on="roi_dir", how="left")

# ───────── compute all_unique_organelles / all_fluor_organelles ─────────

def _split_pipe(val):
    if pd.isna(val):
        return []
    s = str(val).strip()
    if not s:
        return []
    return [p.strip() for p in s.split("|") if p.strip()]

all_unique = []
all_florg  = []

for _, row in df_enrich.iterrows():
    locs = _split_pipe(row.get("genotype_marker_localizations", pd.NA)) + \
           _split_pipe(row.get("treatment_marker_localizations", pd.NA))
    fusions = _split_pipe(row.get("genotype_marker_fusion_labels", pd.NA)) + \
              _split_pipe(row.get("treatment_marker_fluor_loc_labels", pd.NA))

    def uniq(xs):
        out, seen = [], set()
        for x in xs:
            if x not in seen:
                seen.add(x)
                out.append(x)
        return out

    loc_u = uniq(locs)
    fus_u = uniq(fusions)

    all_unique.append("|".join(loc_u) if loc_u else pd.NA)
    all_florg.append("|".join(fus_u) if fus_u else pd.NA)

df_enrich["all_unique_organelles"] = pd.Series(all_unique, dtype="string")
df_enrich["all_fluor_organelles"]  = pd.Series(all_florg,  dtype="string")

# quick sanity sample
print("V8-05 — marker/organelles sample:")
print(
    df_enrich[
        [
            "roi_dir",
            "dataset_slug",
            "genotype_base_codes",
            "treatment_rna_rna_base_code",
            "all_unique_organelles",
            "all_fluor_organelles",
        ]
    ].head(30)
)

V8-05 — marker/organelles sample:
                                              roi_dir                               dataset_slug genotype_base_codes treatment_rna_rna_base_code all_unique_organelles                    all_fluor_organelles
0   /clusterfs/vast/abcabc/Aang_Foundation/2025072...  20250721_72hpf_mrna_mSG_organelle_LLS-SIM                <NA>                        <NA>                  <NA>                                    <NA>
1   /clusterfs/vast/abcabc/Aang_Foundation/2025072...  20250721_72hpf_mrna_mSG_organelle_LLS-SIM                <NA>                        <NA>                  <NA>                                    <NA>
2   /clusterfs/vast/abcabc/Aang_Foundation/2025072...  20250721_72hpf_mrna_mSG_organelle_LLS-SIM                <NA>                        <NA>                  <NA>                                    <NA>
3   /clusterfs/vast/abcabc/Aang_Foundation/2025072...  20250721_72hpf_mrna_mSG_organelle_LLS-SIM                <NA>                      

/var/folders/29/cdrb2nrn01s4d1_n6gvxy5j80000gn/T/ipykernel_39680/2709962229.py:91: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(_agg_geno)
/var/folders/29/cdrb2nrn01s4d1_n6gvxy5j80000gn/T/ipykernel_39680/2709962229.py:153: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(_agg_tx)


In [59]:
# V8-06 — export v8 annotations + DB subset and QC organelles

import pandas as pd
from pathlib import Path

if "df_enrich" not in globals():
    raise NameError("V8-06: df_enrich not found; run V8-01..V8-05 first.")

ROOT = Path("/Users/davekokel/Projects/carp_v2")
BASE = ROOT / "seed_kits" / "legacy_wrangling_v2"
WORKING = BASE / "working"

FULL_OUT_V8 = WORKING / "legacy_imaging_annotations_v8.csv"
DB_OUT_V8   = WORKING / "legacy_imaging_annotations_for_db_v8.csv"

# DB subset columns (no manual overrides, no helper columns)
db_cols = [
    "roi_dir",
    "bruker_roi_id",
    "plate_date",
    "plate_id_filled",
    "slot_id_filled",
    "roi_index_within_slot",
    "dataset_slug",
    "experiment_folder",
    "fish_id",
    "fish_number",
    "fish_age_hpf",
    "roi_anatomy",
    "roi_tiffs",
    "date_experiment",
    "Date imaged",
    "date_mount",
    "genotype_base_codes",
    "genotype_allele_codes",
    "treatment_rna_rna_base_code",
    "treatment_plasmid_plasmid_base_code",
    "all_unique_organelles",
    "all_fluor_organelles",
    "link_source",
]

df_for_db = df_enrich.copy()
for col in db_cols:
    if col not in df_for_db.columns:
        df_for_db[col] = pd.NA

df_for_db = df_for_db[db_cols].copy()

print("V8-06 — df_enrich shape:", df_enrich.shape)
print("V8-06 — unique roi_dir in df_enrich:", df_enrich["roi_dir"].nunique())
print("\nV8-06 — df_for_db shape:", df_for_db.shape)
print("V8-06 — unique roi_dir in df_for_db:", df_for_db["roi_dir"].nunique())

# write outputs
df_enrich.to_csv(FULL_OUT_V8, index=False)
df_for_db.to_csv(DB_OUT_V8, index=False)

print(f"\nV8-06 — wrote FULL_OUT_V8 to: {FULL_OUT_V8}")
print(f"V8-06 — wrote DB_OUT_V8   to: {DB_OUT_V8}")

# QC: organelles
missing_mask = df_for_db["all_unique_organelles"].isna() | (
    df_for_db["all_unique_organelles"].astype(str).str.strip().isin(["", "None", "nan", "<NA>"])
)

n_total   = len(df_for_db)
n_missing = int(missing_mask.sum())
n_present = n_total - n_missing

print("\nV8-06 — organelle coverage (df_for_db):")
print("  total ROIs:              ", n_total)
print("  ROIs with organelles:    ", n_present)
print("  ROIs missing organelles: ", n_missing)

print("\nV8-06 — missing organelles by link_source:")
print(df_for_db.loc[missing_mask, "link_source"].value_counts(dropna=False))

print("\nV8-06 — top datasets among no-org rows:")
print(
    df_for_db.loc[missing_mask, "dataset_slug"]
    .value_counts()
    .head(20)
)

# sanity peek for the must-be-full experiment_hole slugs
must_be_full_slugs = [
    "20250805_lifeact_mem-halo",
    "20251028_mem-peroxi",
    "20251028_mem-peroxi2",
    "20250521_skittles_no-membrane",
    "20251017_nuclear_envelope",
    "20251107_mem-kinectocore",
    "20250602_mem",
    "20251017_microtubules",
]

print("\nV8-06 — MUST-BE-FULL slug coverage:")
print(
    df_for_db[
        df_for_db["dataset_slug"].isin(must_be_full_slugs)
    ][["dataset_slug", "all_unique_organelles"]]
    .groupby("dataset_slug")
    .apply(lambda g: {
        "n_rows": len(g),
        "n_missing": int((g["all_unique_organelles"].isna() | (g["all_unique_organelles"].astype(str).str.strip() == "")).sum())
    })
)

V8-06 — df_enrich shape: (976, 72)
V8-06 — unique roi_dir in df_enrich: 976

V8-06 — df_for_db shape: (976, 23)
V8-06 — unique roi_dir in df_for_db: 976

V8-06 — wrote FULL_OUT_V8 to: /Users/davekokel/Projects/carp_v2/seed_kits/legacy_wrangling_v2/working/legacy_imaging_annotations_v8.csv
V8-06 — wrote DB_OUT_V8   to: /Users/davekokel/Projects/carp_v2/seed_kits/legacy_wrangling_v2/working/legacy_imaging_annotations_for_db_v8.csv

V8-06 — organelle coverage (df_for_db):
  total ROIs:               976
  ROIs with organelles:     929
  ROIs missing organelles:  47

V8-06 — missing organelles by link_source:
link_source
sheet        26
unmatched    21
Name: count, dtype: int64

V8-06 — top datasets among no-org rows:
dataset_slug
20250624_skittlez_Chris_x_Liam               17
Denoising                                    10
20250721_72hpf_mrna_mSG_organelle_LLS-SIM     5
20250805_lifeact_mem-halo                     5
20250624_skittlez_Issac_x_Ken                 5
20251017_nuclear_envelo

/var/folders/29/cdrb2nrn01s4d1_n6gvxy5j80000gn/T/ipykernel_39680/3226988061.py:104: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: {


In [60]:
# V8-06b — rebuild df_for_db WITH basecodes and rewrite DB export

from pathlib import Path
import pandas as pd

if "df_enrich" not in globals():
    raise NameError("V8-06b: df_enrich not found; run V8-01..V8-05 first.")

ROOT = Path("/Users/davekokel/Projects/carp_v2")
BASE = ROOT / "seed_kits" / "legacy_wrangling_v2"
WORKING = BASE / "working"

FULL_OUT_V8 = WORKING / "legacy_imaging_annotations_v8.csv"
DB_OUT_V8   = WORKING / "legacy_imaging_annotations_for_db_v8.csv"

print("V8-06b — FULL_OUT_V8:", FULL_OUT_V8)
print("V8-06b — DB_OUT_V8:  ", DB_OUT_V8)

# Make sure the key basecode / organelle columns exist and are strings
for col in [
    "genotype_base_codes",
    "genotype_allele_codes",
    "treatment_rna_rna_base_code",
    "treatment_plasmid_plasmid_base_code",
    "all_unique_organelles",
    "all_fluor_organelles",
]:
    if col not in df_enrich.columns:
        df_enrich[col] = pd.Series([pd.NA] * len(df_enrich), dtype="string")
    else:
        df_enrich[col] = df_enrich[col].astype("string")

# Columns we want in the DB-facing CSV
db_cols = [
    # structural / identity
    "roi_dir",
    "bruker_roi_id",
    "plate_date",
    "plate_id_filled",
    "slot_id_filled",
    "roi_index_within_slot",
    "dataset_slug",
    "experiment_folder",
    "fish_id",
    "fish_number",
    "fish_age_hpf",
    "roi_anatomy",
    "roi_tiffs",
    "date_experiment",
    "Date imaged",
    "date_mount",
    "link_source",

    # basecodes (genotype + treatments)
    "genotype_base_codes",
    "genotype_allele_codes",
    "treatment_rna_rna_base_code",
    "treatment_plasmid_plasmid_base_code",

    # organelle rollup
    "all_unique_organelles",
    "all_fluor_organelles",
]

# Only keep columns that actually exist (defensive)
db_cols = [c for c in db_cols if c in df_enrich.columns]

df_for_db = df_enrich[db_cols].copy()

print("\nV8-06b — df_for_db shape:", df_for_db.shape)
print("V8-06b — unique roi_dir in df_for_db:", df_for_db["roi_dir"].nunique())

# Write both full and DB CSVs again, using current df_enrich / df_for_db
df_enrich.to_csv(FULL_OUT_V8, index=False)
df_for_db.to_csv(DB_OUT_V8, index=False)

# QC: organelle coverage based on the new df_for_db
missing_mask = df_for_db["all_unique_organelles"].isna() | (
    df_for_db["all_unique_organelles"].astype(str).str.strip().isin(["", "None", "nan", "<NA>"])
)

n_total   = len(df_for_db)
n_missing = int(missing_mask.sum())
n_present = n_total - n_missing

print("\nV8-06b — organelle coverage (df_for_db):")
print("  total ROIs:              ", n_total)
print("  ROIs with organelles:    ", n_present)
print("  ROIs missing organelles: ", n_missing)

print("\nV8-06b — missing organelles by link_source:")
if "link_source" in df_for_db.columns:
    print(df_for_db.loc[missing_mask, "link_source"].value_counts(dropna=False))
else:
    print("  link_source column not in df_for_db")

print("\nV8-06b — sample rows for the nuclear_envelope slug (for sanity):")
slug = "20251017_nuclear_envelope"
print(
    df_for_db[df_for_db["dataset_slug"] == slug][
        [
            "roi_dir",
            "dataset_slug",
            "genotype_base_codes",
            "treatment_rna_rna_base_code",
            "all_unique_organelles",
            "all_fluor_organelles",
        ]
    ].head(20)
)

V8-06b — FULL_OUT_V8: /Users/davekokel/Projects/carp_v2/seed_kits/legacy_wrangling_v2/working/legacy_imaging_annotations_v8.csv
V8-06b — DB_OUT_V8:   /Users/davekokel/Projects/carp_v2/seed_kits/legacy_wrangling_v2/working/legacy_imaging_annotations_for_db_v8.csv

V8-06b — df_for_db shape: (976, 23)
V8-06b — unique roi_dir in df_for_db: 976

V8-06b — organelle coverage (df_for_db):
  total ROIs:               976
  ROIs with organelles:     929
  ROIs missing organelles:  47

V8-06b — missing organelles by link_source:
link_source
sheet        26
unmatched    21
Name: count, dtype: int64

V8-06b — sample rows for the nuclear_envelope slug (for sanity):
                                               roi_dir               dataset_slug genotype_base_codes treatment_rna_rna_base_code all_unique_organelles all_fluor_organelles
448  /clusterfs/vast/abcabc/Aang_Foundation/2025101...  20251017_nuclear_envelope                <NA>                        <NA>                  <NA>                

In [61]:
# V8-07 — basecode-level QC for missing organelles and construct gaps

import pandas as pd
from pathlib import Path
import re

if "df_for_db" not in globals() or "df_enrich" not in globals():
    raise NameError("V8-07: df_for_db / df_enrich not found; run V8-01..V8-06 first.")
if "constructs_ft" not in globals():
    raise NameError("V8-07: constructs_ft not found; run V8-04 first.")

ROOT = Path("/Users/davekokel/Projects/carp_v2")
BASE = ROOT / "seed_kits" / "legacy_wrangling_v2"
WORKING = BASE / "working"

# ───────── 1) identify missing-org rows ─────────

missing_mask = df_for_db["all_unique_organelles"].isna() | (
    df_for_db["all_unique_organelles"].astype(str).str.strip().isin(["", "None", "nan", "<NA>"])
)
df_missing = df_for_db.loc[missing_mask].copy()

print("V8-07 — missing-org rows:", len(df_missing))
print("V8-07 — missing-org datasets:")
print(df_missing["dataset_slug"].value_counts())

# bring in basecode columns from df_enrich
base_cols = [
    "roi_dir",
    "genotype_base_codes",
    "treatment_rna_rna_base_code",
    "treatment_plasmid_plasmid_base_code",
]
df_missing_bc = df_missing.merge(
    df_enrich[base_cols],
    on="roi_dir",
    how="left",
)

def _split_codes(val):
    if pd.isna(val):
        return []
    s = str(val).strip()
    if not s:
        return []
    return [p.strip() for p in re.split(r"[|,]", s) if p.strip()]

# ───────── 2) flatten to ROI×basecode and check constructs_ft coverage ─────────

rows = []
for _, row in df_missing_bc.iterrows():
    ds = row["dataset_slug"]
    roi = row["roi_dir"]
    for src in ["genotype_base_codes", "treatment_rna_rna_base_code", "treatment_plasmid_plasmid_base_code"]:
        codes = _split_codes(row.get(src, pd.NA))
        for code in codes:
            rows.append((ds, roi, src, code))

missing_bc = pd.DataFrame(rows, columns=["dataset_slug", "roi_dir", "source_col", "basecode"])

if missing_bc.empty:
    print("\nV8-07 — no basecodes found on missing-org rows; these are true WT/no-marker cases.")
else:
    constructs_codes = set(
        constructs_ft["plasmid_code"].astype(str).str.strip().unique()
    )

    missing_bc["basecode_norm"] = missing_bc["basecode"].astype(str).str.strip()
    missing_bc["in_constructs"] = missing_bc["basecode_norm"].isin(constructs_codes)

    print("\nV8-07 — basecodes seen on missing-org rows (first 40):")
    print(missing_bc.head(40))

    # summarize which basecodes are not in constructs_ft
    lacking = (
        missing_bc.loc[~missing_bc["in_constructs"]]
        .groupby("basecode_norm")
        .agg(
            n_rows=("roi_dir", "nunique"),
            n_datasets=("dataset_slug", "nunique"),
        )
        .reset_index()
        .sort_values(["n_datasets", "n_rows"], ascending=[False, False])
    )

    print("\nV8-07 — basecodes on missing-org rows that are NOT in constructs_ft:")
    print(lacking)

    # also summarize basecodes that *are* present (so we know which gaps are actually mapping / tag_localization issues)
    present = (
        missing_bc.loc[missing_bc["in_constructs"]]
        .groupby("basecode_norm")
        .agg(
            n_rows=("roi_dir", "nunique"),
            n_datasets=("dataset_slug", "nunique"),
        )
        .reset_index()
        .sort_values(["n_datasets", "n_rows"], ascending=[False, False])
    )

    print("\nV8-07 — basecodes on missing-org rows that ARE in constructs_ft (but still yield no organelle):")
    print(present.head(40))

    # ───────── 3) write a constructs_patch CSV you can use to update constructs_plasmid.csv ─────────

    patch_path = WORKING / "constructs_missing_for_orgs_v8.csv"

    # For basecodes not in constructs_ft at all, we give a stub row with empty fluor/tag;
    # you can fill fluor_code and localization from plasmids_janelia_googlesheet or your notes.
    patch_rows = []
    for _, r in lacking.iterrows():
        patch_rows.append(
            {
                "plasmid_code": r["basecode_norm"],
                "fluor_code": "",
                "tag_code": "",
                "tag_localization": "",
                "n_missing_rois": r["n_rows"],
                "n_missing_datasets": r["n_datasets"],
            }
        )

    patch_df = pd.DataFrame(patch_rows, columns=[
        "plasmid_code",
        "fluor_code",
        "tag_code",
        "tag_localization",
        "n_missing_rois",
        "n_missing_datasets",
    ])

    patch_df.to_csv(patch_path, index=False)
    print(f"\nV8-07 — wrote constructs-level patch template to:\n  {patch_path}")
    print("Fill in fluor_code + tag_localization there, copy those rows into constructs_plasmid.csv,")
    print("then re-run V8-02 → V8-04 → V8-05 → V8-06 to refresh organelles.")

V8-07 — missing-org rows: 47
V8-07 — missing-org datasets:
dataset_slug
20250624_skittlez_Chris_x_Liam               17
Denoising                                    10
20250721_72hpf_mrna_mSG_organelle_LLS-SIM     5
20250805_lifeact_mem-halo                     5
20250624_skittlez_Issac_x_Ken                 5
20251017_nuclear_envelope                     2
20250602_mem                                  1
20251017_microtubules                         1
analysis_test                                 1
Name: count, dtype: int64

V8-07 — no basecodes found on missing-org rows; these are true WT/no-marker cases.
